<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/pnad/RendaDomiciliar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PNAD Contínua — renda domiciliar per capita real

Este notebook é o motor de coleta e publicação da série
`pnad-renda-domiciliar-per-capita-real`.

- **Fonte:** IBGE — Pesquisa Nacional por Amostra de Domicílios Contínua (PNAD Contínua),
  Rendimento de todas as fontes, tabela SIDRA **7533**, variável **10816** — rendimento médio
  mensal real domiciliar per capita, a preços médios do último ano —, classificação **1019 =
  49243** (Total), Brasil (N1).
- **Endpoint:**
  `https://apisidra.ibge.gov.br/values/t/7533/n1/all/v/10816/p/all/c1019/49243?formato=json`
- **Definição:** média, por morador, do rendimento mensal do domicílio — rendimento habitual do
  trabalho das pessoas de 14 anos ou mais mais as outras fontes —, média dos quatro trimestres
  do ano, em reais constantes. É a renda de todos os moradores dividida por eles, não o
  rendimento individual de quem trabalha.
- **Fórmula:** nenhuma — o IBGE já publica a série real (IPCA); o notebook não deflaciona de novo.
- **Unidade e forma:** `R$` inteiro (a fonte publica reais sem centavos), `eixo: reais`,
  `medida: nivel`, `granularidade: anual` (C15: `AAAA-01-01`).
- **Período publicado:** de 2012 (primeiro ano da PNAD Contínua) até o último ano fechado.
- **Arquivo gerado:** `indicadores/pnad/pnad-renda-domiciliar-per-capita-real.json`.

## O que este número não é

- **Não é nominal.** O IBGE já deflaciona a série pelo IPCA e a reexpressa a preços médios do
  último ano a cada divulgação. O notebook não deflaciona de novo; o ano-base do campo `fonte`
  sai do último ano da série, nunca fixo no código.
- **Não é o rendimento individual.** É a renda do domicílio dividida pelo número de moradores;
  o rendimento médio de quem trabalha é outra série (`pnad-rendimento-real`).
- **Não é o número do FPE/TCU.** O valor nominal publicado na imprensa em fevereiro (R$ 2.316
  em 2025) usa outro universo e outro preço; não é esta série real.
- **Não é a tabela 7395.** Aquela tabela (variável 4196) é do módulo TIC, começa em 2016, não
  tem 2020 e dá valores diferentes conforme o ano; foi descartada.
- **Não começa em 1995.** A PNAD Contínua anual começa em 2012; a PNAD antiga (1995–2015) tinha
  outro questionário e outro conceito de renda e não é emendada.


In [1]:
from datetime import date
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "pnad"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("PNAD_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


ID = "pnad-renda-domiciliar-per-capita-real"
TABELA = "7533"
VARIAVEL = "10816"
CLASSIFICACAO = "49243"
URL = (
    "https://apisidra.ibge.gov.br/values/t/7533/n1/all/v/10816/p/all/c1019/49243"
    "?formato=json"
)
PRIMEIRO_ANO = 2012
ANO_CORRENTE = date.today().year


In [2]:
def baixar(url):
    """Busca o recorte completo; repete em erro HTTP, timeout e 200 com HTML."""
    for tentativa in range(1, 6):
        try:
            resposta = requests.get(url, timeout=60)
            resposta.raise_for_status()
            dados = resposta.json()
            if not isinstance(dados, list) or len(dados) < 2:
                raise ValueError("resposta sem linhas de dados")
            if dados[0].get("V") != "Valor":
                raise ValueError(f"cabeçalho inesperado: {dados[0]!r}")
            return dados
        except (requests.RequestException, ValueError):
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def extrair(dados):
    """Devolve {ano: valor} do Total (49243), sem o ano corrente (incompleto)."""
    valores = {}
    for linha in dados[1:]:
        if linha.get("D2C") != VARIAVEL or linha.get("D4C") != CLASSIFICACAO:
            raise ValueError(f"linha fora do recorte: {linha!r}")
        if linha.get("D4N") != "Total":
            raise ValueError(f"classe inesperada: {linha.get('D4N')!r}")
        if linha.get("MN") != "Reais":
            raise ValueError(f"unidade inesperada: {linha.get('MN')!r}")
        nome_variavel = linha.get("D2N", "")
        if "domiciliar per capita" not in nome_variavel or "preços médios" not in nome_variavel:
            raise ValueError(f"variável inesperada: {nome_variavel!r}")
        ano = int(linha["D3C"])
        valor = linha["V"]
        if valor in ("...", "-", "X", ""):
            continue
        valores[ano] = float(valor)
    if not valores:
        raise ValueError("série vazia")
    return {ano: valor for ano, valor in valores.items() if ano < ANO_CORRENTE}


VALORES = extrair(baixar(URL))
ANOS = sorted(VALORES)
ANO_BASE = ANOS[-1]
print(f"{len(ANOS)} anos fechados: {ANOS[0]} a {ANO_BASE}")


14 anos fechados: 2012 a 2025


In [3]:
fonte = (
    "IBGE (PNAD Contínua — Rendimento de todas as fontes, SIDRA "
    f"{TABELA}, variável {VARIAVEL}; a preços médios de {ANO_BASE})"
)

pontos = [
    {"data": f"{ano}-01-01", "valor": VALORES[ano]}  # C15: anual = AAAA-01-01
    for ano in ANOS
]

serie = {
    "versao": 1,
    "id": ID,
    "unidade": {"tipo": "moeda", "simbolo": "R$", "casas": 0},
    "eixo": "reais",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": fonte,
    "urlFonte": URL,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
caminho = DESTINO / f"{ID}.json"
with caminho.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)
print(f"gravado {relativo(caminho)}: {len(pontos)} pontos, {ANOS[0]} a {ANO_BASE}")


gravado indicadores/pnad/pnad-renda-domiciliar-per-capita-real.json: 14 pontos, 2012 a 2025


## Validação inline

A célula falha se o contrato, a ordem, a unicidade, a contiguidade anual, a finitude, a
igualdade com a fonte ou o ano-base do campo `fonte` forem violados. A conferência é contra a
fonte; os números citados na metodologia são conferidos no PR, não aqui.


In [4]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
UNIDADE = {"tipo": "moeda", "simbolo": "R$", "casas": 0}

assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == ID, serie["id"]
assert serie["unidade"] == UNIDADE, serie["unidade"]
assert serie["eixo"] == "reais", serie["eixo"]
assert serie["medida"] == "nivel", serie["medida"]
assert serie["granularidade"] == "anual", serie["granularidade"]
assert serie["urlFonte"] == URL, serie["urlFonte"]
assert serie["fonte"].endswith(f"a preços médios de {ANO_BASE})"), serie["fonte"]

dados = serie["dados"]
assert dados, "série vazia"
datas = [ponto["data"] for ponto in dados]
dias = [date.fromisoformat(texto) for texto in datas]
for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["data"], str), ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto
    assert ponto["valor"] > 0, ponto

# C15: anual no primeiro dia do ano.
assert all(dia.month == 1 and dia.day == 1 for dia in dias), "data fora de AAAA-01-01"
assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem"
assert all(b.year - a.year == 1 for a, b in zip(dias, dias[1:])), "ano faltando ou repetido"
assert datas[0] == f"{PRIMEIRO_ANO}-01-01", datas[0]
assert datas[-1] == f"{ANO_BASE}-01-01", datas[-1]
assert len(dados) == ANO_BASE - PRIMEIRO_ANO + 1, len(dados)
assert len(dados) == len(VALORES), (len(dados), len(VALORES))

esperado = [
    {"data": f"{ano}-01-01", "valor": VALORES[ano]}
    for ano in ANOS
]
assert dados == esperado, "valor publicado difere da fonte"

with caminho.open(encoding="utf-8") as arquivo:
    assert json.load(arquivo) == serie, "arquivo gravado difere do objeto"

print(
    f"ok: {len(dados)} anos, {datas[0]} a {datas[-1]}; "
    f"série idêntica à fonte, a preços médios de {ANO_BASE}"
)


ok: 14 anos, 2012-01-01 a 2025-01-01; série idêntica à fonte, a preços médios de 2025
